
# NiriKsha — Font Size & Readability Analysis
### AI/ML Feature Development & Evaluation Notebook (SIH 2026)

**Scope of this notebook (and only this scope):**

- **Font Size Analysis** — estimating the *physical* height (in mm) of printed
  declaration text, but **only** when a valid physical calibration is available.
- **Readability Analysis** — assessing the *visual quality* of a printed text
  region (blur, contrast, sharpness, OCR confidence, resolution, etc.), which is
  **not the same thing** as "OCR was able to read it".

**Explicitly out of scope here** (built elsewhere in NiriKsha):
mobile app, FastAPI backend, legal-rule engine, report generation, and any
final **legal compliance decision**. This notebook produces *measurements and
uncertainty-aware model outputs only* — e.g. `ESTIMATED`, `READABLE`,
`LOW_READABILITY`, `FONT_SIZE_UNDETERMINABLE`, `MANUAL_VERIFICATION_REQUIRED`.
It never emits a verdict like `"LEGAL VIOLATION"`.

**Data honesty policy used throughout this notebook:**

1. No legal thresholds are hard-coded anywhere in this notebook. Every legal
   threshold (minimum font height in mm, minimum readability standard, etc.)
   lives in NiriKsha's separate, verified rule engine — not here.
2. This notebook ships with a **tiny synthetic/demo dataset** only, so that
   every cell is runnable end-to-end in a fresh Colab runtime. It is labeled
   `DEMONSTRATION DATA — NOT REAL-WORLD TRAINING DATA` everywhere it appears,
   and no metric produced from it should ever be reported as real model
   performance. Section 3 (Dataset Structure) and Section 22-equivalent cells
   explain exactly how to swap in real, human-annotated package images.
3. OCR is **modular**: a `run_ocr()` function defines the interface, with a
   pluggable backend. For the demo we use a `mock` backend (so the notebook
   does not depend on downloading large OCR model weights in a fresh runtime),
   but the exact same downstream pipeline accepts real bounding boxes /
   confidences exported by NiriKsha's existing OCR component, or a real
   PaddleOCR/EasyOCR backend (stubs provided, commented out).

### Conceptual pipeline

```
Package Image
      |
Text Detection / OCR
      |
Text Bounding Boxes
      |
Crop Individual Text Regions
      |
   +--+--+
   |     |
   v     v
Font Size   Readability
Analysis    Analysis
   |             |
Physical      Readability
Text Height    Score
   |             |
   +------+------+
          |
   Combined Analysis (this notebook stops HERE)
          |
ESTIMATED / READABLE / LOW_READABILITY /
UNDETERMINABLE / MANUAL_VERIFICATION_REQUIRED
          |
 (handed to NiriKsha rule engine — NOT this notebook)
```



## 2. Environment Setup

Installs / imports for classical ML + computer-vision. Deep learning and
heavy OCR engines (PaddleOCR / EasyOCR) are **optional** and left as
commented stubs — see the note in the OCR section — so this notebook stays
lightweight, reproducible, and does not depend on downloading large model
weights just to demonstrate the pipeline.


In [ ]:

# ----------------------------------------------------------------------
# 2.1 Install dependencies (safe to re-run; Colab already has most of these)
# ----------------------------------------------------------------------
import sys, subprocess

REQUIRED = [
    "numpy", "pandas", "scikit-learn", "opencv-python-headless",
    "matplotlib", "seaborn", "joblib", "Pillow", "scipy",
]

def pip_install(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

pip_install(REQUIRED)

# XGBoost is optional (nice-to-have for a stronger gradient-boosted baseline).
try:
    import xgboost  # noqa: F401
    HAS_XGBOOST = True
except ImportError:
    pip_install(["xgboost"])
    try:
        import xgboost  # noqa: F401
        HAS_XGBOOST = True
    except ImportError:
        HAS_XGBOOST = False

print("XGBoost available:", HAS_XGBOOST)


In [ ]:

# ----------------------------------------------------------------------
# 2.2 Optional real OCR backends (NOT installed/run by default)
# ----------------------------------------------------------------------
# This notebook's OCR layer is modular (see Section: "Text Detection / OCR").
# In a real NiriKsha run you would either:
#   (a) load bounding boxes + confidences already produced by NiriKsha's
#       existing OCR component (preferred — avoids duplicating work), or
#   (b) run a real OCR engine here.
#
# Uncomment ONE of the following if you want to run real OCR in this notebook.
# They are left commented out because model-weight downloads are slow/heavy
# and not needed to develop and evaluate the Font Size / Readability models
# themselves (those operate on bounding boxes + image crops, not on OCR
# internals).
#
# pip_install(["paddlepaddle", "paddleocr"])
# from paddleocr import PaddleOCR
# PADDLE_OCR_ENGINE = PaddleOCR(use_angle_cls=True, lang="en")
#
# pip_install(["easyocr"])
# import easyocr
# EASY_OCR_ENGINE = easyocr.Reader(["en"])
print("Real OCR backends left uninstalled by default (see comments above).")


In [ ]:

# ----------------------------------------------------------------------
# 2.3 Imports
# ----------------------------------------------------------------------
import os
import json
import random
import warnings
import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import joblib

from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import (
    RandomForestRegressor, RandomForestClassifier,
    GradientBoostingRegressor, GradientBoostingClassifier,
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score, median_absolute_error,
    accuracy_score, precision_recall_fscore_support, confusion_matrix,
    classification_report, roc_curve, auc, precision_recall_curve,
)

if HAS_XGBOOST:
    from xgboost import XGBRegressor, XGBClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

print("All libraries imported successfully.")


In [ ]:

# ----------------------------------------------------------------------
# 2.4 CPU / GPU check
# ----------------------------------------------------------------------
# This notebook is deliberately classical-ML / CV based (Section 25 of the
# spec: "Do not over-engineer this... only introduce deep learning if the
# dataset is sufficiently large and there is a demonstrated reason to do
# so"). A GPU is therefore NOT required. We still report what's available,
# since a future deep-learning extension (e.g. a CNN readability model once
# thousands of real annotated crops exist) would benefit from it.
import platform

print("Python:", platform.python_version())
print("Platform:", platform.platform())

gpu_available = False
try:
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    gpu_available = result.returncode == 0
    if gpu_available:
        print(result.stdout)
except FileNotFoundError:
    pass

print("GPU available:", gpu_available, "(not required for this notebook)")


In [ ]:

# ----------------------------------------------------------------------
# 2.5 Google Drive (optional)
# ----------------------------------------------------------------------
# Only useful once you have REAL package images + annotations stored in
# Drive. Safe to skip when running the demo pipeline end-to-end.
MOUNT_DRIVE = False  # set True in Colab if you have a real dataset on Drive

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        print("Google Drive mounted.")
    except ImportError:
        print("Not running in Colab — skipping Drive mount.")
else:
    print("MOUNT_DRIVE=False — skipping Google Drive mount.")


In [ ]:

# ----------------------------------------------------------------------
# 2.6 Reproducibility
# ----------------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print(f"Random seed fixed at {SEED}")


In [ ]:

# ----------------------------------------------------------------------
# 2.7 Directory structure
# ----------------------------------------------------------------------
BASE_DIR = Path("./nirikSha_font_readability")
DATA_DIR = BASE_DIR / "data"
IMAGES_DIR = DATA_DIR / "images"                 # package / crop images
DEMO_DIR = DATA_DIR / "demo_synthetic"           # DEMO DATA lives ONLY here
ANNOTATIONS_DIR = DATA_DIR / "annotations"
MODELS_DIR = BASE_DIR / "models"
REPORTS_DIR = BASE_DIR / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"

for d in [DATA_DIR, IMAGES_DIR, DEMO_DIR, ANNOTATIONS_DIR, MODELS_DIR,
          REPORTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Project directories ready:")
for d in [BASE_DIR, DATA_DIR, IMAGES_DIR, DEMO_DIR, ANNOTATIONS_DIR,
          MODELS_DIR, REPORTS_DIR, FIGURES_DIR]:
    print(" -", d)



## 3. Dataset Structure

Each row of the core dataset represents **one detected text region** (not one
whole package image). The required schema, exactly as specified for
NiriKsha's Font Size & Readability feature:

| Column | Meaning |
|---|---|
| `image_id` | unique id of the text-region crop |
| `package_id` | unique id of the **package** the region came from (multiple regions share a package_id) |
| `image_path` | path to the package image (or crop) |
| `bbox_x1,bbox_y1,bbox_x2,bbox_y2` | OCR/detector bounding box, in pixels, on the *original* image |
| `text` | OCR'd (or ground-truth, for annotation) text string |
| `declaration_type` | e.g. `MRP`, `NET_QUANTITY`, `MFG_DATE`, `CONSUMER_CARE`, `GENERIC` — free text, NOT a legal classification |
| `ocr_confidence` | OCR engine's own confidence score, 0–1 |
| `actual_font_height_mm` | **ground-truth** physical font height, only present when (a) a human/reference measurement exists AND (b) valid pixel-to-mm calibration exists for that image |
| `readability_label` | one of `READABLE` / `PARTIALLY_READABLE` / `NOT_READABLE` — **must come from human annotation on real data** (Section: Human Annotation). Never invent this for real images. |
| `readability_score` | optional continuous 0–1 score, also human-sourced where available |

### How real data must be collected (do this before trusting any metric below)

1. Photograph real packaged-commodity labels under varied, realistic
   conditions (different distances, angles, lighting, phone cameras).
2. For **font size** ground truth: physically measure a sample of printed
   characters with calipers/a ruler on the actual package, **or** include a
   known-size reference object/marker in frame for calibration (Section:
   Physical Calibration). Record `actual_font_height_mm` only for those
   measured samples.
3. For **readability** ground truth: have a human annotator view each cropped
   text region (Section: Human Annotation workflow below) and assign
   `READABLE` / `PARTIALLY_READABLE` / `NOT_READABLE` based on whether *they*
   can read it comfortably — not based on whether OCR succeeded.
4. Store everything in the CSV schema above, with images referenced by
   `image_path`.

### About the data used to *run* this notebook

Because no real annotated package-image dataset is attached to this
notebook yet, Section 4 below generates a **tiny synthetic/demo dataset**
purely so every cell executes and produces sane, inspectable output. It is
tagged `DEMONSTRATION DATA — NOT REAL-WORLD TRAINING DATA` and none of its
metrics are meaningful measures of real-world model performance. Swap in a
real CSV following the schema above (and real images under `IMAGES_DIR`) to
get meaningful results — the rest of the pipeline does not need to change.



## 4. Data Loading

`load_dataset(csv_path)` is the single entry point the rest of the notebook
uses. Point it at a real annotations CSV (matching the Section 3 schema) to
use real data. Below we instead **synthesize a tiny demo dataset** so the
notebook can run standalone.

> ⚠️ **DEMONSTRATION DATA — NOT REAL-WORLD TRAINING DATA.**
> Text, images, "OCR" boxes, calibration, and readability degradations below
> are all synthetically generated. `actual_font_height_mm` values are
> synthetic ground truth *for demo purposes only* (computed from the exact
> pixel font size we asked PIL to draw, divided by a synthetic
> pixels-per-mm we assigned per package to simulate different camera
> distances) — they are **not** measurements of any real product.


In [ ]:

# ----------------------------------------------------------------------
# 4.1 Synthetic package-image generator (DEMO DATA ONLY)
# ----------------------------------------------------------------------
N_DEMO_PACKAGES = 18

DECLARATION_TYPES = ["MRP", "NET_QUANTITY", "MFG_DATE", "CONSUMER_CARE", "GENERIC"]

SAMPLE_TEXTS = {
    "MRP": ["MRP RS 199.00", "MRP RS 45.50 INCL. OF ALL TAXES", "M.R.P RS 1250"],
    "NET_QUANTITY": ["NET WT 500 g", "NET QTY 1 L", "NET WEIGHT 250 GRAMS"],
    "MFG_DATE": ["MFG DATE 03/2026", "PKD ON 11-2025", "BEST BEFORE 12/2027"],
    "CONSUMER_CARE": ["CUSTOMER CARE 1800-000-000", "FOR COMPLAINTS CALL HELPLINE"],
    "GENERIC": ["INGREDIENTS: WHEAT, SUGAR, SALT", "STORE IN A COOL DRY PLACE",
                "MANUFACTURED BY XYZ FOODS PVT LTD"],
}

# A real TrueType font ships with matplotlib -> reuse it so text renders
# crisply at arbitrary pixel sizes (Pillow's default bitmap font does not
# scale cleanly).
FONT_PATH = fm.findfont(fm.FontProperties(family="DejaVu Sans"))


def _degrade_image(pil_img, mode):
    '''Apply a controlled visual-quality degradation. SYNTHETIC AUGMENTATION
    only -- used to create readability variation for the demo, and later
    (Section: Image Quality Experiments) for explicit robustness testing.
    Never a substitute for real human-labelled readability data.'''
    arr = np.array(pil_img.convert("RGB"))
    if mode == "none":
        return arr
    if mode == "blur":
        k = random.choice([3, 5, 7])
        return cv2.GaussianBlur(arr, (k, k), 0)
    if mode == "low_contrast":
        mean = arr.mean()
        return np.clip(mean + (arr - mean) * 0.25, 0, 255).astype(np.uint8)
    if mode == "noise":
        noise = np.random.normal(0, 18, arr.shape)
        return np.clip(arr + noise, 0, 255).astype(np.uint8)
    if mode == "downscale":
        h, w = arr.shape[:2]
        small = cv2.resize(arr, (max(4, w // 4), max(4, h // 4)))
        return cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)
    if mode == "jpeg":
        ok, enc = cv2.imencode(".jpg", arr, [cv2.IMWRITE_JPEG_QUALITY, 15])
        return cv2.imdecode(enc, cv2.IMREAD_COLOR)
    return arr


def generate_demo_package(package_id, rng):
    '''Generates ONE synthetic package label image with several printed
    declaration lines at KNOWN pixel font sizes, and a KNOWN synthetic
    pixels-per-mm calibration factor for that package (simulating the
    package being photographed at a different distance each time).'''
    W, H = rng.choice([900, 1100, 1300]), rng.choice([600, 700, 900])
    bg_options = [(250, 248, 240), (255, 255, 255), (240, 240, 235)]
    bg = tuple(int(c) for c in bg_options[rng.integers(0, len(bg_options))])
    img = Image.new("RGB", (int(W), int(H)), bg)
    draw = ImageDraw.Draw(img)

    # Simulate a per-package camera distance -> pixels-per-mm.
    # (Synthetic stand-in for a real calibration marker / known package
    # dimension — see Section: Physical Calibration for the REAL method.)
    true_pixels_per_mm = rng.uniform(6.0, 14.0)
    has_calibration = rng.random() > 0.2  # ~20% of demo packages: NO calibration

    n_lines = rng.integers(2, 5)
    y_cursor = int(0.08 * H)
    rows = []
    for _ in range(n_lines):
        decl_type = rng.choice(DECLARATION_TYPES)
        text = rng.choice(SAMPLE_TEXTS[decl_type])
        font_px = int(rng.integers(14, 46))  # drawn font size, in pixels
        font = ImageFont.truetype(FONT_PATH, font_px)

        x = int(0.06 * W)
        color = tuple(int(c) for c in rng.integers(10, 60, size=3))
        draw.text((x, y_cursor), text, font=font, fill=color)

        bbox = draw.textbbox((x, y_cursor), text, font=font)  # (x1,y1,x2,y2)
        rows.append(dict(
            text=text, declaration_type=decl_type,
            bbox=bbox, font_px=font_px,
        ))
        y_cursor += font_px + int(0.03 * H)
        if y_cursor > H - 20:
            break

    return img, rows, true_pixels_per_mm, has_calibration


def build_demo_dataset(n_packages=N_DEMO_PACKAGES, seed=SEED):
    rng = np.random.default_rng(seed)
    degrade_modes = ["none", "none", "blur", "low_contrast", "noise",
                      "downscale", "jpeg"]
    records = []
    calibration_records = []

    for i in range(n_packages):
        package_id = f"PKG_{i:03d}"
        img, rows, px_per_mm, has_cal = generate_demo_package(package_id, rng)

        mode = rng.choice(degrade_modes)
        degraded = _degrade_image(img, mode)
        img_path = DEMO_DIR / f"{package_id}.png"
        Image.fromarray(degraded).save(img_path)

        calibration_records.append(dict(
            package_id=package_id,
            pixels_per_mm=float(px_per_mm) if has_cal else None,
            calibration_source="synthetic_demo_reference_object" if has_cal else None,
            calibration_valid=bool(has_cal),
        ))

        for j, r in enumerate(rows):
            x1, y1, x2, y2 = r["bbox"]
            # Simulate OCR bounding-box imprecision (real detectors are not
            # pixel-perfect) so the pixel->mm baseline is not trivially
            # exact and there's a genuine regression target for the ML
            # models to improve on.
            jitter = rng.normal(0, 1.2, size=4)
            x1j, y1j, x2j, y2j = (np.array([x1, y1, x2, y2]) + jitter).clip(0)

            ocr_conf = float(np.clip(rng.normal(0.9 if mode == "none" else 0.72, 0.08), 0.05, 0.99))

            actual_mm = (
                round((y2 - y1) / px_per_mm, 3) if has_cal else None
            )

            # Synthetic readability rule (DEMO ONLY — NOT a human annotation).
            # Combines the applied degradation + font size as a rough proxy
            # so the demo classifier has *some* real, non-random signal in
            # this synthetic run; a real project MUST replace this with
            # human-annotated labels (see Section: Human Annotation).
            severity = {"none": 0, "downscale": 1, "low_contrast": 1,
                        "jpeg": 2, "noise": 2, "blur": 3}[mode]
            small_text_penalty = 1 if r["font_px"] < 20 else 0
            score = max(0.0, 1.0 - 0.22 * severity - 0.15 * small_text_penalty
                        + rng.normal(0, 0.05))
            score = float(np.clip(score, 0, 1))
            if score >= 0.7:
                label = "READABLE"
            elif score >= 0.4:
                label = "PARTIALLY_READABLE"
            else:
                label = "NOT_READABLE"

            records.append(dict(
                image_id=f"{package_id}_R{j:02d}",
                package_id=package_id,
                image_path=str(img_path),
                bbox_x1=round(float(x1j), 1), bbox_y1=round(float(y1j), 1),
                bbox_x2=round(float(x2j), 1), bbox_y2=round(float(y2j), 1),
                text=r["text"],
                declaration_type=r["declaration_type"],
                ocr_confidence=round(ocr_conf, 3),
                actual_font_height_mm=actual_mm,
                readability_label=label,
                readability_score=round(score, 3),
                _demo_degrade_mode=mode,             # demo-only debug column
                _demo_true_font_px=r["font_px"],      # demo-only debug column
            ))

    df = pd.DataFrame(records)
    cal_df = pd.DataFrame(calibration_records)
    return df, cal_df


demo_df, demo_calibration_df = build_demo_dataset()
print(f"DEMONSTRATION DATA — NOT REAL-WORLD TRAINING DATA")
print(f"Generated {len(demo_df)} text regions across "
      f"{demo_df['package_id'].nunique()} synthetic packages.")
demo_df.drop(columns=["_demo_degrade_mode", "_demo_true_font_px"]).head(8)


In [ ]:

# ----------------------------------------------------------------------
# 4.2 load_dataset(): the ONE function the rest of the notebook calls
# ----------------------------------------------------------------------
REQUIRED_COLUMNS = [
    "image_id", "package_id", "image_path",
    "bbox_x1", "bbox_y1", "bbox_x2", "bbox_y2",
    "text", "declaration_type", "ocr_confidence",
    "actual_font_height_mm", "readability_label", "readability_score",
]


def load_dataset(csv_path=None, dataframe=None):
    '''Loads the text-region dataset.

    - Pass `csv_path` to a real annotations CSV (Section 3 schema) for real
      data.
    - Pass `dataframe` directly (used here to plug in the demo dataset).
    Performs schema validation (Section: Dataset Validation) before returning.
    '''
    if dataframe is not None:
        df = dataframe.copy()
    elif csv_path is not None:
        df = pd.read_csv(csv_path)
    else:
        raise ValueError("Provide either csv_path or dataframe")

    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Dataset missing required columns: {missing}")
    return df


dataset_df = load_dataset(dataframe=demo_df)
print("Loaded dataset shape:", dataset_df.shape)



## 5. Annotation Format

Annotations are stored as plain CSV/JSON so any team member can review and
edit them without special tooling, and so they merge cleanly with the
`REQUIRED_COLUMNS` schema from Section 3.

**Font-size annotation record** (one per manually measured text region):
```json
{
  "image_id": "PKG_014_R02",
  "package_id": "PKG_014",
  "actual_font_height_mm": 2.3,
  "measurement_method": "caliper | reference_marker | known_package_dimension",
  "annotator_id": "A01",
  "annotation_notes": "measured on physical package, good lighting"
}
```

**Readability annotation record** (one per human-reviewed crop):
```json
{
  "image_id": "PKG_014_R02",
  "package_id": "PKG_014",
  "readability_label": "PARTIALLY_READABLE",
  "readability_score": 0.55,
  "annotator_id": "A01",
  "annotation_quality": "high | medium | low",
  "annotation_notes": "glare on right edge of text"
}
```

Both record types are merged into the master CSV keyed by `image_id`. The
interactive annotation tool in Section: Human Annotation writes exactly
these records.



## 6. Dataset Validation

Basic structural / sanity checks — run this on **any** dataset (demo or
real) before splitting or training.


In [ ]:

def validate_dataset(df):
    issues = []

    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        issues.append(f"Missing columns: {missing}")

    if df["image_id"].duplicated().any():
        dups = df.loc[df["image_id"].duplicated(), "image_id"].tolist()
        issues.append(f"Duplicate image_id values: {dups[:5]}...")

    bad_bbox = df[
        (df["bbox_x2"] <= df["bbox_x1"]) | (df["bbox_y2"] <= df["bbox_y1"])
    ]
    if len(bad_bbox):
        issues.append(f"{len(bad_bbox)} rows have an invalid bbox "
                       f"(x2<=x1 or y2<=y1)")

    bad_conf = df[(df["ocr_confidence"] < 0) | (df["ocr_confidence"] > 1)]
    if len(bad_conf):
        issues.append(f"{len(bad_conf)} rows have ocr_confidence outside [0,1]")

    missing_images = [p for p in df["image_path"].unique() if not Path(p).exists()]
    if missing_images:
        issues.append(f"{len(missing_images)} image_path(s) not found on disk")

    valid_labels = {"READABLE", "PARTIALLY_READABLE", "NOT_READABLE", None, np.nan}
    bad_labels = df[~df["readability_label"].isin(valid_labels) &
                     df["readability_label"].notna()]
    if len(bad_labels):
        issues.append(f"{len(bad_labels)} rows have an unrecognised readability_label")

    n_font_labeled = df["actual_font_height_mm"].notna().sum()
    n_read_labeled = df["readability_label"].notna().sum()

    report = {
        "n_rows": len(df),
        "n_packages": df["package_id"].nunique(),
        "n_font_size_labeled_rows": int(n_font_labeled),
        "n_readability_labeled_rows": int(n_read_labeled),
        "issues": issues,
    }
    return report


validation_report = validate_dataset(dataset_df)
print(json.dumps(validation_report, indent=2))
assert not validation_report["issues"], "Fix dataset issues before continuing."
print("\\nDataset passed structural validation.")



## 7. Package-Level Data Split (leakage prevention)

Multiple text regions come from the **same package**. Splitting by
`image_id` would let regions from one package leak across train/val/test
(the model could implicitly "recognise" a package it partly saw in
training). We therefore split by **`package_id`**, ~70/15/15, and verify
afterwards that no package appears in more than one split.


In [ ]:

def package_level_split(df, train_frac=0.70, val_frac=0.15, seed=SEED):
    test_frac = 1.0 - train_frac - val_frac
    assert abs(train_frac + val_frac + test_frac - 1.0) < 1e-9

    groups = df["package_id"].values

    gss1 = GroupShuffleSplit(n_splits=1, train_size=train_frac, random_state=seed)
    train_idx, rest_idx = next(gss1.split(df, groups=groups))

    rest_df = df.iloc[rest_idx]
    rel_val_frac = val_frac / (val_frac + test_frac)
    gss2 = GroupShuffleSplit(n_splits=1, train_size=rel_val_frac, random_state=seed)
    val_idx_rel, test_idx_rel = next(
        gss2.split(rest_df, groups=rest_df["package_id"].values)
    )

    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df = rest_df.iloc[val_idx_rel].reset_index(drop=True)
    test_df = rest_df.iloc[test_idx_rel].reset_index(drop=True)
    return train_df, val_df, test_df


train_df, val_df, test_df = package_level_split(dataset_df)

print(f"Train: {len(train_df)} regions / {train_df['package_id'].nunique()} packages")
print(f"Val:   {len(val_df)} regions / {val_df['package_id'].nunique()} packages")
print(f"Test:  {len(test_df)} regions / {test_df['package_id'].nunique()} packages")


In [ ]:

# ----------------------------------------------------------------------
# Leakage verification report — MUST show zero overlap
# ----------------------------------------------------------------------
train_pkgs = set(train_df["package_id"])
val_pkgs = set(val_df["package_id"])
test_pkgs = set(test_df["package_id"])

overlap_tv = train_pkgs & val_pkgs
overlap_tt = train_pkgs & test_pkgs
overlap_vt = val_pkgs & test_pkgs

print("=== Package-level leakage verification report ===")
print(f"Train ∩ Val  packages: {len(overlap_tv)}  {'OK' if not overlap_tv else 'LEAK!!'}")
print(f"Train ∩ Test packages: {len(overlap_tt)}  {'OK' if not overlap_tt else 'LEAK!!'}")
print(f"Val   ∩ Test packages: {len(overlap_vt)}  {'OK' if not overlap_vt else 'LEAK!!'}")

assert not overlap_tv and not overlap_tt and not overlap_vt, \
    "Data leakage detected across splits!"
print("\\nNo package-level leakage detected across train/val/test splits.")



## Text Detection / OCR (modular layer)

The Font Size and Readability models both consume **bounding boxes + image
crops**, not OCR internals. `run_ocr()` below defines a single interface with
pluggable backends:

- `backend="from_columns"` — (used below) trusts bounding boxes already
  present in the dataset (this is how NiriKsha's existing OCR component's
  output should be loaded — no need to re-run OCR here).
- `backend="mock"` — for quick ad-hoc testing on a raw image with no prior
  boxes.
- `backend="paddleocr"` / `"easyocr"` — real engines, left as stubs (Section
  2.2) so this notebook doesn't require downloading OCR model weights just to
  develop/evaluate the Font Size & Readability models.


In [ ]:

def run_ocr(image=None, df_row=None, backend="from_columns"):
    '''Returns a list of dicts: [{bbox: (x1,y1,x2,y2), text, ocr_confidence}, ...]

    backend="from_columns": trust bbox/text/confidence already in df_row
        (this is the path used for both the demo dataset and any real
        NiriKsha OCR export loaded via load_dataset()).
    backend="mock": crude fallback OCR-like output for a raw image with no
        prior boxes (NOT for production use).
    backend="paddleocr"/"easyocr": placeholders for real engines (Section 2.2).
    '''
    if backend == "from_columns":
        assert df_row is not None, "from_columns backend requires df_row"
        return [{
            "bbox": (df_row["bbox_x1"], df_row["bbox_y1"],
                     df_row["bbox_x2"], df_row["bbox_y2"]),
            "text": df_row["text"],
            "ocr_confidence": df_row["ocr_confidence"],
        }]
    elif backend == "mock":
        assert image is not None, "mock backend requires an image"
        h, w = image.shape[:2]
        return [{"bbox": (int(0.05*w), int(0.05*h), int(0.6*w), int(0.2*h)),
                  "text": "", "ocr_confidence": 0.0}]
    elif backend in ("paddleocr", "easyocr"):
        raise NotImplementedError(
            f"'{backend}' backend requires the optional engine from "
            f"Section 2.2 to be installed and instantiated."
        )
    else:
        raise ValueError(f"Unknown OCR backend: {backend}")


def crop_text_region(image_bgr, bbox, pad_ratio=0.06):
    '''Crops a bounding box from an image with a small safety padding.'''
    x1, y1, x2, y2 = [int(round(v)) for v in bbox]
    h, w = image_bgr.shape[:2]
    pad_x = int((x2 - x1) * pad_ratio)
    pad_y = int((y2 - y1) * pad_ratio)
    x1, y1 = max(0, x1 - pad_x), max(0, y1 - pad_y)
    x2, y2 = min(w, x2 + pad_x), min(h, y2 + pad_y)
    return image_bgr[y1:y2, x1:x2]


def load_image_bgr(path):
    img = cv2.imread(str(path))
    if img is None:
        raise FileNotFoundError(f"Could not read image: {path}")
    return img


# Sanity check on one demo row
_sample_row = dataset_df.iloc[0]
_sample_img = load_image_bgr(_sample_row["image_path"])
_ocr_out = run_ocr(df_row=_sample_row, backend="from_columns")
_crop = crop_text_region(_sample_img, _ocr_out[0]["bbox"])
print("OCR output:", _ocr_out)
print("Crop shape:", _crop.shape)



# PART A — FONT SIZE ANALYSIS

## 8. Font Size Features

Purely geometric / textual features derived from the bounding box, the
crop, and the OCR text — no physical units yet (that only happens after
valid calibration is applied, next section).


In [ ]:

def extract_font_size_features(row, image_bgr=None):
    '''Feature vector for font-size estimation. Returns a flat dict.'''
    x1, y1, x2, y2 = row["bbox_x1"], row["bbox_y1"], row["bbox_x2"], row["bbox_y2"]
    pixel_width = max(x2 - x1, 1e-6)
    pixel_height = max(y2 - y1, 1e-6)

    if image_bgr is None:
        image_bgr = load_image_bgr(row["image_path"])
    img_h, img_w = image_bgr.shape[:2]

    text = str(row["text"]) if pd.notna(row["text"]) else ""
    char_count = len(text.replace(" ", ""))
    word_count = max(len(text.split()), 1)
    text_length = len(text)
    avg_char_width = pixel_width / max(char_count, 1)

    crop = crop_text_region(image_bgr, (x1, y1, x2, y2))
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY) if crop.size else np.zeros((1, 1), np.uint8)
    sharpness = float(cv2.Laplacian(gray, cv2.CV_64F).var()) if gray.size else 0.0
    contrast = float(gray.std()) if gray.size else 0.0

    return {
        "pixel_height": pixel_height,
        "pixel_width": pixel_width,
        "aspect_ratio": pixel_width / pixel_height,
        "normalized_height": pixel_height / img_h,
        "normalized_width": pixel_width / img_w,
        "image_width": img_w,
        "image_height": img_h,
        "text_length": text_length,
        "char_count": char_count,
        "word_count": word_count,
        "avg_char_width": avg_char_width,
        "ocr_confidence": float(row["ocr_confidence"]),
        "crop_resolution": crop.shape[0] * crop.shape[1] if crop.size else 0,
        "sharpness": sharpness,
        "contrast": contrast,
    }


FONT_SIZE_FEATURE_NAMES = [
    "pixel_height", "pixel_width", "aspect_ratio", "normalized_height",
    "normalized_width", "image_width", "image_height", "text_length",
    "char_count", "word_count", "avg_char_width", "ocr_confidence",
    "crop_resolution", "sharpness", "contrast",
]

print("Extracting font-size features for all rows (cached by image)...")
_img_cache = {}

def _get_cached_image(path):
    if path not in _img_cache:
        _img_cache[path] = load_image_bgr(path)
    return _img_cache[path]

font_feat_rows = []
for _, r in dataset_df.iterrows():
    img = _get_cached_image(r["image_path"])
    font_feat_rows.append(extract_font_size_features(r, img))

font_features_df = pd.DataFrame(font_feat_rows)
font_features_df["image_id"] = dataset_df["image_id"].values
font_features_df["package_id"] = dataset_df["package_id"].values
font_features_df.head()



## 9. Physical Calibration

This is the most legally-sensitive part of Font Size Analysis, so it gets
explicit, conservative handling:

- We **never** assume image DPI/EXIF metadata gives a true physical scale —
  DPI in a photo is typically meaningless for real-world size (it reflects
  the file format's stored resolution, not distance-to-subject).
- We **never** predict a millimeter value without a *validated* pixels-per-mm
  calibration for that specific image/package.
- If calibration is missing or fails validation, the correct output is the
  explicit status `FONT_SIZE_UNDETERMINABLE` — not a guess.

Valid calibration sources (any one is acceptable, in order of preference for
real deployment):

1. **Known reference object** placed in frame at capture time (e.g. a
   calibration card / ruler of known size).
2. **Known package dimension** — e.g. package width already known from
   product metadata, matched against its pixel width in the photo.
3. **Calibration marker** printed/attached specifically for inspection.
4. **Manually supplied pixels-per-mm** — an inspector enters it directly
   (e.g. from a known fixed camera rig with a fixed focal distance).

`validate_calibration()` below only checks *numerical plausibility*
(positive, finite, within a broad, physically-sane range for close-up
product photography) — it is **not** a legal threshold and must not be
confused with one.


In [ ]:

# NOTE: this numeric range is a plausibility/sanity check on the
# calibration itself (is 1 pixel physically between roughly 0.02mm and 5mm?),
# NOT a legal minimum font size. Legal thresholds live only in NiriKsha's
# rule engine, never in this notebook.
_PLAUSIBLE_PIXELS_PER_MM_RANGE = (0.2, 200.0)


def calculate_pixels_per_mm(measured_pixel_distance, known_physical_distance_mm):
    '''pixels_per_mm = measured_pixel_distance / known_physical_distance_mm'''
    if known_physical_distance_mm is None or known_physical_distance_mm <= 0:
        return None
    if measured_pixel_distance is None or measured_pixel_distance <= 0:
        return None
    return measured_pixel_distance / known_physical_distance_mm


def validate_calibration(pixels_per_mm):
    '''Numerical plausibility check only -- NOT a legal threshold.'''
    if pixels_per_mm is None:
        return False, "no calibration provided"
    if not np.isfinite(pixels_per_mm) or pixels_per_mm <= 0:
        return False, "calibration value is non-finite or non-positive"
    lo, hi = _PLAUSIBLE_PIXELS_PER_MM_RANGE
    if not (lo <= pixels_per_mm <= hi):
        return False, f"calibration {pixels_per_mm:.3f} px/mm outside plausible range {_PLAUSIBLE_PIXELS_PER_MM_RANGE}"
    return True, "ok"


def pixel_to_mm(pixel_value, pixels_per_mm):
    valid, _ = validate_calibration(pixels_per_mm)
    if not valid:
        return None
    return pixel_value / pixels_per_mm


def estimate_font_height(pixel_height, calibration_info):
    '''calibration_info: dict with at least {"pixels_per_mm": float or None}.
    Returns a status-carrying dict -- never a bare number -- so downstream
    consumers cannot silently mistake an undetermined case for a real
    measurement.'''
    pixels_per_mm = calibration_info.get("pixels_per_mm")
    valid, reason = validate_calibration(pixels_per_mm)
    if not valid:
        return {"status": "FONT_SIZE_UNDETERMINABLE", "height_mm": None, "reason": reason}
    return {
        "status": "ESTIMATED",
        "height_mm": round(pixel_to_mm(pixel_height, pixels_per_mm), 3),
        "reason": "valid calibration",
        "calibration_source": calibration_info.get("calibration_source"),
    }


# Demo sanity checks
print(estimate_font_height(40, {"pixels_per_mm": 10.0, "calibration_source": "demo"}))
print(estimate_font_height(40, {"pixels_per_mm": None}))
print(estimate_font_height(40, {"pixels_per_mm": -3}))


In [ ]:

# Attach calibration info (from demo_calibration_df, i.e. what a real
# per-package calibration table would look like) onto the feature table.
font_features_df = font_features_df.merge(
    demo_calibration_df, on="package_id", how="left"
)

calibration_outcomes = font_features_df.apply(
    lambda r: estimate_font_height(
        r["pixel_height"],
        {"pixels_per_mm": r["pixels_per_mm"], "calibration_source": r["calibration_source"]},
    ),
    axis=1,
)
font_features_df["calibration_status"] = calibration_outcomes.apply(lambda d: d["status"])
font_features_df["height_mm_from_calibration"] = calibration_outcomes.apply(lambda d: d["height_mm"])

print(font_features_df["calibration_status"].value_counts())
font_features_df[["package_id", "pixels_per_mm", "calibration_status",
                   "height_mm_from_calibration"]].drop_duplicates("package_id").head(10)



## 10. Font Size Baseline

The simplest possible baseline is the calibration equation itself:
`font_height_mm = pixel_height / pixels_per_mm`. We evaluate this baseline
**only on rows with valid calibration and real ground truth**
(`actual_font_height_mm` present) — never on undetermined rows.

Because the demo dataset injects small, realistic pixel-level bounding-box
jitter (Section 4), this baseline is not perfect even in the synthetic
setting, which gives the ML models in Section 11 something real to improve
on.


In [ ]:

font_labeled_df = font_features_df.merge(
    dataset_df[["image_id", "actual_font_height_mm", "declaration_type"]],
    on="image_id", how="left", suffixes=("", "_gt"),
)
# actual_font_height_mm came through twice (once via dataset merge earlier
# indirectly) -- keep a single clean ground-truth column.
font_labeled_df = font_labeled_df.rename(columns={"actual_font_height_mm": "actual_font_height_mm_gt"}) \
    if "actual_font_height_mm_gt" not in font_labeled_df.columns else font_labeled_df

font_eval_df = font_labeled_df[
    font_labeled_df["calibration_status"].eq("ESTIMATED") &
    font_labeled_df["actual_font_height_mm_gt"].notna()
].copy()

print(f"Rows usable for font-size baseline/model evaluation "
      f"(valid calibration + ground truth): {len(font_eval_df)} / {len(font_labeled_df)}")

font_eval_df["baseline_pred_mm"] = font_eval_df["height_mm_from_calibration"]

baseline_mae = mean_absolute_error(font_eval_df["actual_font_height_mm_gt"], font_eval_df["baseline_pred_mm"])
baseline_rmse = mean_squared_error(font_eval_df["actual_font_height_mm_gt"], font_eval_df["baseline_pred_mm"]) ** 0.5
baseline_r2 = r2_score(font_eval_df["actual_font_height_mm_gt"], font_eval_df["baseline_pred_mm"])
baseline_medae = median_absolute_error(font_eval_df["actual_font_height_mm_gt"], font_eval_df["baseline_pred_mm"])

print(f"BASELINE (pixel_height / pixels_per_mm) -- on DEMO data only:")
print(f"  MAE:  {baseline_mae:.4f} mm")
print(f"  RMSE: {baseline_rmse:.4f} mm")
print(f"  R2:   {baseline_r2:.4f}")
print(f"  MedAE:{baseline_medae:.4f} mm")



## 11. Font Size ML Models

We train lightweight regressors on top of the Section 8 features to predict
`actual_font_height_mm`, and compare them against the Section 10 baseline.
Only rows with **valid calibration + real ground truth** are used (same
`font_eval_df` filter). The package-level train/val/test split from Section
7 is reused so packages never leak across sets.

> Reminder: on the demo dataset, "real ground truth" means the synthetic
> ground truth described in Section 4 — useful for proving the pipeline
> works end-to-end, not for claiming real-world accuracy.


In [ ]:

font_train_ids = set(train_df["image_id"])
font_val_ids = set(val_df["image_id"])
font_test_ids = set(test_df["image_id"])

fs_train = font_eval_df[font_eval_df["image_id"].isin(font_train_ids)].copy()
fs_val = font_eval_df[font_eval_df["image_id"].isin(font_val_ids)].copy()
fs_test = font_eval_df[font_eval_df["image_id"].isin(font_test_ids)].copy()

print(f"Font-size model rows -- train: {len(fs_train)}, val: {len(fs_val)}, test: {len(fs_test)}")

X_train, y_train = fs_train[FONT_SIZE_FEATURE_NAMES], fs_train["actual_font_height_mm_gt"]
X_val, y_val = fs_val[FONT_SIZE_FEATURE_NAMES], fs_val["actual_font_height_mm_gt"]
X_test, y_test = fs_test[FONT_SIZE_FEATURE_NAMES], fs_test["actual_font_height_mm_gt"]


In [ ]:

font_size_models = {
    "LinearRegression": LinearRegression(),
    "RandomForestRegressor": RandomForestRegressor(n_estimators=200, max_depth=6, random_state=SEED),
    "GradientBoostingRegressor": GradientBoostingRegressor(random_state=SEED),
}
if HAS_XGBOOST:
    font_size_models["XGBRegressor"] = XGBRegressor(
        n_estimators=200, max_depth=4, learning_rate=0.08,
        random_state=SEED, verbosity=0,
    )

font_size_scaler = StandardScaler().fit(X_train)
X_train_sc = font_size_scaler.transform(X_train)
X_val_sc = font_size_scaler.transform(X_val)

font_size_results = []
fitted_font_models = {}

for name, model in font_size_models.items():
    use_scaled = isinstance(model, LinearRegression)
    Xtr = X_train_sc if use_scaled else X_train
    Xv = X_val_sc if use_scaled else X_val

    model.fit(Xtr, y_train)
    val_pred = model.predict(Xv)

    font_size_results.append({
        "model": name,
        "val_MAE": mean_absolute_error(y_val, val_pred),
        "val_RMSE": mean_squared_error(y_val, val_pred) ** 0.5,
        "val_R2": r2_score(y_val, val_pred),
        "val_MedAE": median_absolute_error(y_val, val_pred),
    })
    fitted_font_models[name] = model

font_size_results_df = pd.DataFrame(font_size_results).sort_values("val_MAE")
print("Validation-set comparison (DEMO data):")
font_size_results_df


In [ ]:

# Select the best model by validation MAE (never by test-set performance).
best_font_model_name = font_size_results_df.iloc[0]["model"]
best_font_model = fitted_font_models[best_font_model_name]
print(f"Selected font-size model: {best_font_model_name}")



## 12. Font Size Cross Validation

`GroupKFold` with `group = package_id`, so folds never split a package's
text regions across train/validation within CV either. Run on
train+validation combined (test stays untouched).


In [ ]:

cv_pool_font = pd.concat([fs_train, fs_val], ignore_index=True)
X_cv, y_cv, groups_cv = (
    cv_pool_font[FONT_SIZE_FEATURE_NAMES],
    cv_pool_font["actual_font_height_mm_gt"],
    cv_pool_font["package_id"],
)

n_groups = groups_cv.nunique()
n_splits = min(5, n_groups)
gkf = GroupKFold(n_splits=n_splits)

from sklearn.base import clone

cv_scores = {"MAE": [], "RMSE": [], "R2": []}
for fold, (tr_idx, te_idx) in enumerate(gkf.split(X_cv, y_cv, groups=groups_cv)):
    model = clone(best_font_model)
    model.fit(X_cv.iloc[tr_idx], y_cv.iloc[tr_idx])
    pred = model.predict(X_cv.iloc[te_idx])
    cv_scores["MAE"].append(mean_absolute_error(y_cv.iloc[te_idx], pred))
    cv_scores["RMSE"].append(mean_squared_error(y_cv.iloc[te_idx], pred) ** 0.5)
    cv_scores["R2"].append(r2_score(y_cv.iloc[te_idx], pred))
    print(f"Fold {fold}: MAE={cv_scores['MAE'][-1]:.4f}  RMSE={cv_scores['RMSE'][-1]:.4f}  R2={cv_scores['R2'][-1]:.4f}")

print(f"\n{n_splits}-fold GroupKFold (group=package_id), model={best_font_model_name}:")
for metric, vals in cv_scores.items():
    print(f"  {metric}: mean={np.mean(vals):.4f}  std={np.std(vals):.4f}")



## 13. Font Size Evaluation (held-out test set)

Final, one-time evaluation of the selected model on the untouched test
split, compared against the naive calibration-only baseline.


In [ ]:

use_scaled = isinstance(best_font_model, LinearRegression)
X_test_eval = font_size_scaler.transform(X_test) if use_scaled else X_test
test_pred = best_font_model.predict(X_test_eval)

font_size_final_metrics = {
    "model": best_font_model_name,
    "test_MAE": mean_absolute_error(y_test, test_pred),
    "test_RMSE": mean_squared_error(y_test, test_pred) ** 0.5,
    "test_R2": r2_score(y_test, test_pred),
    "test_MedAE": median_absolute_error(y_test, test_pred),
    "n_test_rows": int(len(y_test)),
}
baseline_test = fs_test["baseline_pred_mm"]
baseline_final_metrics = {
    "model": "Baseline (pixel/calibration)",
    "test_MAE": mean_absolute_error(y_test, baseline_test),
    "test_RMSE": mean_squared_error(y_test, baseline_test) ** 0.5,
    "test_R2": r2_score(y_test, baseline_test),
    "test_MedAE": median_absolute_error(y_test, baseline_test),
    "n_test_rows": int(len(y_test)),
}

font_size_comparison_df = pd.DataFrame([baseline_final_metrics, font_size_final_metrics])
print("Test-set comparison -- DEMO DATA, illustrative only:")
font_size_comparison_df


In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Actual vs predicted
axes[0, 0].scatter(y_test, test_pred, alpha=0.7, edgecolor="k")
lims = [min(y_test.min(), test_pred.min()), max(y_test.max(), test_pred.max())]
axes[0, 0].plot(lims, lims, "r--", label="perfect prediction")
axes[0, 0].set_xlabel("Actual font height (mm)")
axes[0, 0].set_ylabel("Predicted font height (mm)")
axes[0, 0].set_title(f"Actual vs Predicted -- {best_font_model_name}")
axes[0, 0].legend()

# Residuals
residuals = test_pred - y_test.values
axes[0, 1].scatter(test_pred, residuals, alpha=0.7, edgecolor="k")
axes[0, 1].axhline(0, color="r", linestyle="--")
axes[0, 1].set_xlabel("Predicted font height (mm)")
axes[0, 1].set_ylabel("Residual (pred - actual)")
axes[0, 1].set_title("Residual Plot")

# Error distribution
axes[1, 0].hist(np.abs(residuals), bins=15, edgecolor="k", alpha=0.8)
axes[1, 0].set_xlabel("Absolute error (mm)")
axes[1, 0].set_ylabel("Count")
axes[1, 0].set_title("Absolute Error Distribution")

# Target distribution
axes[1, 1].hist(dataset_df["actual_font_height_mm"].dropna(), bins=15, edgecolor="k", alpha=0.8, color="orange")
axes[1, 1].set_xlabel("Font height (mm)")
axes[1, 1].set_ylabel("Count")
axes[1, 1].set_title("Target Distribution (all labeled rows)")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "font_size_evaluation.png", dpi=120)
plt.show()


In [ ]:

# Feature importance (tree-based models only)
if hasattr(best_font_model, "feature_importances_"):
    importances = pd.Series(best_font_model.feature_importances_, index=FONT_SIZE_FEATURE_NAMES)
    importances = importances.sort_values(ascending=True)
    plt.figure(figsize=(8, 6))
    importances.plot(kind="barh")
    plt.title(f"Feature Importance -- {best_font_model_name}")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "font_size_feature_importance.png", dpi=120)
    plt.show()
else:
    coefs = pd.Series(best_font_model.coef_, index=FONT_SIZE_FEATURE_NAMES).sort_values()
    plt.figure(figsize=(8, 6))
    coefs.plot(kind="barh", color="teal")
    plt.title(f"Standardized Coefficients -- {best_font_model_name}")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "font_size_coefficients.png", dpi=120)
    plt.show()



## 14. Font Size Error Analysis

Inspect the worst-performing predictions to understand *why* the model
failed — small text, low OCR confidence, unusual aspect ratio, etc.


In [ ]:

font_error_df = fs_test[["image_id", "package_id", "declaration_type", "ocr_confidence"]].copy()
font_error_df["actual_mm"] = y_test.values
font_error_df["predicted_mm"] = test_pred
font_error_df["abs_error_mm"] = np.abs(font_error_df["predicted_mm"] - font_error_df["actual_mm"])
font_error_df = font_error_df.sort_values("abs_error_mm", ascending=False)

print("Worst 10 font-size predictions (DEMO data):")
font_error_df.head(10)



# PART B — READABILITY ANALYSIS

## 15. Readability Features

Readability is a **visual quality** property of a text region, distinct from
whether OCR happened to succeed on it. We compute a broad set of measurable
visual-quality features per crop, and let the readability model combine
them — no single metric is treated as sufficient on its own.


In [ ]:

def _edge_density(gray):
    edges = cv2.Canny(gray, 60, 150)
    return float((edges > 0).mean())


def _noise_estimate(gray):
    # Difference between the image and a median-blurred version approximates
    # high-frequency noise not explained by legitimate edges/text strokes.
    denoised = cv2.medianBlur(gray, 3)
    return float(np.mean(np.abs(gray.astype(np.float32) - denoised.astype(np.float32))))


def _text_background_separation(gray):
    # Otsu thresholding splits the crop into two intensity clusters; a well
    # separated (readable) text region should show a large gap between the
    # two cluster means.
    if gray.size == 0 or gray.std() == 0:
        return 0.0
    _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    fg = gray[mask == 255]
    bg = gray[mask == 0]
    if fg.size == 0 or bg.size == 0:
        return 0.0
    return float(abs(fg.mean() - bg.mean()))


def _perspective_distortion_indicator(gray):
    # Crude, measurable proxy: how non-rectangular the strongest text-like
    # contour is. This is intentionally simple -- a full perspective/skew
    # estimator belongs in the OCR/detection layer, not here.
    edges = cv2.Canny(gray, 60, 150)
    coords = cv2.findNonZero(edges)
    if coords is None or len(coords) < 5:
        return 0.0
    rect = cv2.minAreaRect(coords)
    (_, _), (rw, rh), angle = rect
    return float(min(abs(angle), abs(90 - abs(angle))))  # 0 = axis aligned


def extract_readability_features(row, image_bgr=None):
    '''Feature vector for readability assessment. Returns a flat dict.'''
    if image_bgr is None:
        image_bgr = load_image_bgr(row["image_path"])
    crop = crop_text_region(image_bgr, (row["bbox_x1"], row["bbox_y1"], row["bbox_x2"], row["bbox_y2"]))
    if crop.size == 0:
        crop = np.zeros((4, 4, 3), dtype=np.uint8)
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)

    laplacian_var = float(cv2.Laplacian(gray, cv2.CV_64F).var())  # sharpness
    contrast_std = float(gray.std())
    edge_density = _edge_density(gray)
    noise_est = _noise_estimate(gray)
    text_bg_sep = _text_background_separation(gray)
    perspective_ind = _perspective_distortion_indicator(gray)

    crop_h, crop_w = gray.shape[:2]
    text = str(row["text"]) if pd.notna(row["text"]) else ""
    char_count = max(len(text.replace(" ", "")), 1)
    pixels_per_char = (crop_w * crop_h) / char_count

    # "Blur score": inverse relationship to sharpness, scaled 0-1 (0=sharp).
    blur_score = float(1.0 / (1.0 + laplacian_var / 100.0))

    return {
        "sharpness_laplacian_var": laplacian_var,
        "contrast_std": contrast_std,
        "edge_density": edge_density,
        "grayscale_mean": float(gray.mean()),
        "grayscale_std": contrast_std,
        "noise_estimate": noise_est,
        "text_background_separation": text_bg_sep,
        "perspective_distortion_indicator": perspective_ind,
        "ocr_confidence": float(row["ocr_confidence"]),
        "crop_width": crop_w,
        "crop_height": crop_h,
        "crop_resolution": crop_w * crop_h,
        "pixels_per_char": pixels_per_char,
        "char_density": char_count / max(crop_w * crop_h, 1),
        "blur_score": blur_score,
    }


READABILITY_FEATURE_NAMES = [
    "sharpness_laplacian_var", "contrast_std", "edge_density", "grayscale_mean",
    "grayscale_std", "noise_estimate", "text_background_separation",
    "perspective_distortion_indicator", "ocr_confidence", "crop_width",
    "crop_height", "crop_resolution", "pixels_per_char", "char_density",
    "blur_score",
]

print("Extracting readability features for all rows...")
read_feat_rows = []
for _, r in dataset_df.iterrows():
    img = _get_cached_image(r["image_path"])
    read_feat_rows.append(extract_readability_features(r, img))

readability_features_df = pd.DataFrame(read_feat_rows)
readability_features_df["image_id"] = dataset_df["image_id"].values
readability_features_df["package_id"] = dataset_df["package_id"].values
readability_features_df["readability_label"] = dataset_df["readability_label"].values
readability_features_df["readability_score"] = dataset_df["readability_score"].values
readability_features_df["bbox_x1"] = dataset_df["bbox_x1"].values
readability_features_df["bbox_y1"] = dataset_df["bbox_y1"].values
readability_features_df["bbox_x2"] = dataset_df["bbox_x2"].values
readability_features_df["bbox_y2"] = dataset_df["bbox_y2"].values
readability_features_df["image_path"] = dataset_df["image_path"].values
readability_features_df.head()



## 16. Image Quality Baselines

Before any ML: simple, single-metric thresholds (blur / contrast / OCR
confidence). These are **CV heuristics for comparison purposes only** — they
are explicitly *not* asserted to be legally valid readability standards.


In [ ]:

def cv_baseline_readability(features, blur_thresh=80.0, contrast_thresh=25.0, ocr_conf_thresh=0.6):
    '''Simple rule-based baseline -- NOT a legal standard, comparison only.
    Returns one of READABLE / PARTIALLY_READABLE / NOT_READABLE.'''
    votes_ok = 0
    votes_total = 3
    if features["sharpness_laplacian_var"] >= blur_thresh:
        votes_ok += 1
    if features["contrast_std"] >= contrast_thresh:
        votes_ok += 1
    if features["ocr_confidence"] >= ocr_conf_thresh:
        votes_ok += 1

    if votes_ok == votes_total:
        return "READABLE"
    elif votes_ok == 0:
        return "NOT_READABLE"
    else:
        return "PARTIALLY_READABLE"


readability_features_df["cv_baseline_label"] = readability_features_df.apply(
    lambda r: cv_baseline_readability(r), axis=1
)

cv_baseline_acc = accuracy_score(
    readability_features_df["readability_label"], readability_features_df["cv_baseline_label"]
)
print(f"CV-only baseline accuracy on full DEMO dataset: {cv_baseline_acc:.3f}")
print(readability_features_df[["readability_label", "cv_baseline_label"]].value_counts())



## 17. Readability ML Models

Trained on visual + OCR features to predict `readability_label`, using the
same package-level train/val/test split. Because this is an **inspection**
system, a false `READABLE` prediction (the model says a genuinely
unreadable label is fine) is worse than a false `NOT_READABLE`/
`PARTIALLY_READABLE` prediction — the latter merely routes to manual
review, while the former could let a non-compliant label pass unnoticed.
We therefore report **per-class recall for `NOT_READABLE`/
`PARTIALLY_READABLE`** alongside overall accuracy, not accuracy alone.


In [ ]:

read_train = readability_features_df[readability_features_df["image_id"].isin(font_train_ids)].copy()
read_val = readability_features_df[readability_features_df["image_id"].isin(font_val_ids)].copy()
read_test = readability_features_df[readability_features_df["image_id"].isin(font_test_ids)].copy()

print(f"Readability model rows -- train: {len(read_train)}, val: {len(read_val)}, test: {len(read_test)}")

LABEL_ORDER = ["NOT_READABLE", "PARTIALLY_READABLE", "READABLE"]

Xr_train, yr_train = read_train[READABILITY_FEATURE_NAMES], read_train["readability_label"]
Xr_val, yr_val = read_val[READABILITY_FEATURE_NAMES], read_val["readability_label"]
Xr_test, yr_test = read_test[READABILITY_FEATURE_NAMES], read_test["readability_label"]

readability_scaler = StandardScaler().fit(Xr_train)
Xr_train_sc = readability_scaler.transform(Xr_train)
Xr_val_sc = readability_scaler.transform(Xr_val)


In [ ]:

readability_models = {
    "LogisticRegression": LogisticRegression(max_iter=2000),
    "RandomForestClassifier": RandomForestClassifier(n_estimators=250, max_depth=6, random_state=SEED),
    "GradientBoostingClassifier": GradientBoostingClassifier(random_state=SEED),
}
if HAS_XGBOOST:
    _label_to_int = {l: i for i, l in enumerate(LABEL_ORDER)}
    readability_models["XGBClassifier"] = XGBClassifier(
        n_estimators=250, max_depth=4, learning_rate=0.08,
        random_state=SEED, verbosity=0, eval_metric="mlogloss",
    )

readability_results = []
fitted_readability_models = {}

for name, model in readability_models.items():
    use_scaled = isinstance(model, LogisticRegression)
    Xtr = Xr_train_sc if use_scaled else Xr_train
    Xv = Xr_val_sc if use_scaled else Xr_val

    if name == "XGBClassifier":
        model.fit(Xtr, yr_train.map(_label_to_int))
        val_pred_int = model.predict(Xv)
        val_pred = pd.Series(val_pred_int).map({v: k for k, v in _label_to_int.items()}).values
    else:
        model.fit(Xtr, yr_train)
        val_pred = model.predict(Xv)

    acc = accuracy_score(yr_val, val_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        yr_val, val_pred, labels=LABEL_ORDER, average=None, zero_division=0
    )
    not_readable_recall = rec[LABEL_ORDER.index("NOT_READABLE")]
    partially_readable_recall = rec[LABEL_ORDER.index("PARTIALLY_READABLE")]

    readability_results.append({
        "model": name,
        "val_accuracy": acc,
        "val_macro_f1": f1.mean(),
        "val_recall_NOT_READABLE": not_readable_recall,
        "val_recall_PARTIALLY_READABLE": partially_readable_recall,
    })
    fitted_readability_models[name] = model

readability_results_df = pd.DataFrame(readability_results).sort_values(
    "val_recall_NOT_READABLE", ascending=False
)
print("Validation-set comparison (DEMO data) -- sorted to prioritise catching "
      "unreadable text, not just overall accuracy:")
readability_results_df


In [ ]:

best_readability_model_name = readability_results_df.iloc[0]["model"]
best_readability_model = fitted_readability_models[best_readability_model_name]
print(f"Selected readability model: {best_readability_model_name}")
print()
print("Why prioritise NOT_READABLE / PARTIALLY_READABLE recall over raw accuracy:")
print("- A false 'READABLE' prediction means a genuinely hard-to-read declaration")
print("  could be waved through without manual review -- a silent failure mode.")
print("- A false 'NOT_READABLE'/'PARTIALLY_READABLE' prediction just sends a")
print("  perfectly fine label to manual review -- extra work, but no missed issue.")
print("- Accuracy alone can hide a model that is great on the easy majority class")
print("  (READABLE) while quietly missing the minority, safety-relevant classes.")



## 18. Readability Cross Validation

`GroupKFold` with `group = package_id`, same rationale as Section 12.


In [ ]:

cv_pool_read = pd.concat([read_train, read_val], ignore_index=True)
Xr_cv, yr_cv, groups_cv_r = (
    cv_pool_read[READABILITY_FEATURE_NAMES],
    cv_pool_read["readability_label"],
    cv_pool_read["package_id"],
)

n_groups_r = groups_cv_r.nunique()
n_splits_r = min(5, n_groups_r)
gkf_r = GroupKFold(n_splits=n_splits_r)

cv_scores_r = {"accuracy": [], "macro_f1": [], "recall_NOT_READABLE": []}
for fold, (tr_idx, te_idx) in enumerate(gkf_r.split(Xr_cv, yr_cv, groups=groups_cv_r)):
    model = clone(best_readability_model)
    if best_readability_model_name == "XGBClassifier":
        model.fit(Xr_cv.iloc[tr_idx], yr_cv.iloc[tr_idx].map(_label_to_int))
        pred_int = model.predict(Xr_cv.iloc[te_idx])
        pred = pd.Series(pred_int).map({v: k for k, v in _label_to_int.items()}).values
    else:
        model.fit(Xr_cv.iloc[tr_idx], yr_cv.iloc[tr_idx])
        pred = model.predict(Xr_cv.iloc[te_idx])

    y_true_fold = yr_cv.iloc[te_idx]
    acc = accuracy_score(y_true_fold, pred)
    _, rec, f1, _ = precision_recall_fscore_support(
        y_true_fold, pred, labels=LABEL_ORDER, average=None, zero_division=0
    )
    cv_scores_r["accuracy"].append(acc)
    cv_scores_r["macro_f1"].append(f1.mean())
    cv_scores_r["recall_NOT_READABLE"].append(rec[LABEL_ORDER.index("NOT_READABLE")])
    print(f"Fold {fold}: accuracy={acc:.3f}  macro_f1={f1.mean():.3f}  "
          f"recall(NOT_READABLE)={cv_scores_r['recall_NOT_READABLE'][-1]:.3f}")

print(f"\n{n_splits_r}-fold GroupKFold (group=package_id), model={best_readability_model_name}:")
for metric, vals in cv_scores_r.items():
    print(f"  {metric}: mean={np.mean(vals):.3f}  std={np.std(vals):.3f}")



## 19. Readability Evaluation (held-out test set)

Final, one-time evaluation on the untouched test split: accuracy,
precision/recall/F1 per class, confusion matrix, and ROC/PR curves for the
`NOT_READABLE` class (the safety-relevant one). Also compares against the
Section 16 CV-only baseline to check whether ML is actually adding value.


In [ ]:

if best_readability_model_name == "XGBClassifier":
    Xr_test_eval = Xr_test
    test_pred_int = best_readability_model.predict(Xr_test_eval)
    read_test_pred = pd.Series(test_pred_int).map({v: k for k, v in _label_to_int.items()}).values
else:
    Xr_test_eval = readability_scaler.transform(Xr_test) if isinstance(best_readability_model, LogisticRegression) else Xr_test
    read_test_pred = best_readability_model.predict(Xr_test_eval)

print("=== ML Readability Model -- Test Set Classification Report ===")
print(classification_report(yr_test, read_test_pred, labels=LABEL_ORDER, zero_division=0))

cv_baseline_test_acc = accuracy_score(read_test["readability_label"], read_test["cv_baseline_label"])
ml_test_acc = accuracy_score(yr_test, read_test_pred)
print(f"CV-only baseline test accuracy: {cv_baseline_test_acc:.3f}")
print(f"ML model ({best_readability_model_name}) test accuracy: {ml_test_acc:.3f}")
print("(DEMO DATA -- illustrative comparison only, not a real-world accuracy claim.)")


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(yr_test, read_test_pred, labels=LABEL_ORDER)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=LABEL_ORDER,
            yticklabels=LABEL_ORDER, ax=axes[0])
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")
axes[0].set_title(f"Confusion Matrix -- {best_readability_model_name}")

readability_features_df["readability_label"].value_counts().reindex(LABEL_ORDER).plot(
    kind="bar", ax=axes[1], color=["#d62728", "#ff7f0e", "#2ca02c"]
)
axes[1].set_title("Class Distribution (full demo dataset)")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "readability_confusion_and_distribution.png", dpi=120)
plt.show()


In [ ]:

# ROC / Precision-Recall for the safety-relevant NOT_READABLE class
# (one-vs-rest), where the model supports probability estimates.
if hasattr(best_readability_model, "predict_proba"):
    proba = best_readability_model.predict_proba(Xr_test_eval)
    class_order = list(best_readability_model.classes_)
    if best_readability_model_name == "XGBClassifier":
        class_order = [{v: k for k, v in _label_to_int.items()}[c] for c in class_order]
    idx_not_readable = class_order.index("NOT_READABLE")
    y_true_bin = (yr_test.values == "NOT_READABLE").astype(int)
    scores = proba[:, idx_not_readable]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    fpr, tpr, _ = roc_curve(y_true_bin, scores)
    roc_auc = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    axes[0].plot([0, 1], [0, 1], "k--", alpha=0.4)
    axes[0].set_xlabel("False Positive Rate")
    axes[0].set_ylabel("True Positive Rate")
    axes[0].set_title("ROC -- NOT_READABLE vs rest")
    axes[0].legend()

    prec, rec, _ = precision_recall_curve(y_true_bin, scores)
    axes[1].plot(rec, prec)
    axes[1].set_xlabel("Recall")
    axes[1].set_ylabel("Precision")
    axes[1].set_title("Precision-Recall -- NOT_READABLE vs rest")

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "readability_roc_pr.png", dpi=120)
    plt.show()

    print(f"NOTE: these are the model's own predicted probabilities, calibrated "
          f"only to the extent {best_readability_model_name} calibrates them. "
          f"They are model scores, not verified real-world probabilities.")
else:
    print(f"{best_readability_model_name} does not expose predict_proba -- "
          f"skipping ROC/PR curves (would require an uncalibrated decision "
          f"function instead, which we avoid presenting as a probability).")


In [ ]:

# Feature importance / coefficients for the readability model
if hasattr(best_readability_model, "feature_importances_"):
    importances = pd.Series(best_readability_model.feature_importances_, index=READABILITY_FEATURE_NAMES)
    importances = importances.sort_values(ascending=True)
    plt.figure(figsize=(8, 6))
    importances.plot(kind="barh", color="darkorange")
    plt.title(f"Feature Importance -- {best_readability_model_name}")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "readability_feature_importance.png", dpi=120)
    plt.show()
elif hasattr(best_readability_model, "coef_"):
    coefs = pd.DataFrame(
        best_readability_model.coef_, columns=READABILITY_FEATURE_NAMES, index=best_readability_model.classes_
    )
    plt.figure(figsize=(9, 6))
    sns.heatmap(coefs, cmap="coolwarm", center=0, annot=False)
    plt.title(f"Standardized Coefficients -- {best_readability_model_name}")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "readability_coefficients.png", dpi=120)
    plt.show()


In [ ]:

plt.figure(figsize=(7, 5))
sns.histplot(readability_features_df["readability_score"].dropna(), bins=15, kde=True)
plt.xlabel("Readability score (0-1)")
plt.title("Readability Score Distribution (full demo dataset)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "readability_score_distribution.png", dpi=120)
plt.show()



## 20. Readability Error Analysis

Worst-performing examples: cases where the model disagreed with the label,
alongside the crop, OCR confidence, blur, and contrast -- to understand
*why*. False `READABLE` predictions (predicted READABLE, actual
NOT_READABLE or PARTIALLY_READABLE) are called out separately since they
are the higher-risk error type (Section 17).


In [ ]:

read_error_df = read_test[["image_id", "package_id", "bbox_x1", "bbox_y1", "bbox_x2", "bbox_y2",
                            "sharpness_laplacian_var", "contrast_std", "ocr_confidence", "image_path"]].copy()
read_error_df["actual_label"] = yr_test.values
read_error_df["predicted_label"] = read_test_pred
if hasattr(best_readability_model, "predict_proba"):
    read_error_df["model_score_NOT_READABLE"] = scores

read_error_df["is_wrong"] = read_error_df["actual_label"] != read_error_df["predicted_label"]
read_error_df["is_false_readable"] = (
    (read_error_df["predicted_label"] == "READABLE") & (read_error_df["actual_label"] != "READABLE")
)

print(f"Total test errors: {read_error_df['is_wrong'].sum()} / {len(read_error_df)}")
print(f"High-risk false-READABLE errors: {read_error_df['is_false_readable'].sum()}")

worst_examples = read_error_df[read_error_df["is_wrong"]].sort_values(
    "is_false_readable", ascending=False
)
worst_examples.head(10)


In [ ]:

# Visualise a few of the worst (especially false-READABLE) examples with their crops.
n_show = min(4, worst_examples["is_false_readable"].sum() or len(worst_examples))
show_df = pd.concat([
    worst_examples[worst_examples["is_false_readable"]],
    worst_examples[~worst_examples["is_false_readable"]],
]).head(n_show)

if len(show_df):
    fig, axes = plt.subplots(1, len(show_df), figsize=(4 * len(show_df), 4))
    if len(show_df) == 1:
        axes = [axes]
    for ax, (_, row) in zip(axes, show_df.iterrows()):
        img = load_image_bgr(row["image_path"])
        crop = crop_text_region(img, (row["bbox_x1"], row["bbox_y1"], row["bbox_x2"], row["bbox_y2"]))
        ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        tag = "FALSE-READABLE" if row["is_false_readable"] else "misclassified"
        ax.set_title(f"{tag}\nactual={row['actual_label']}\npred={row['predicted_label']}", fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "readability_worst_examples.png", dpi=120)
    plt.show()
else:
    print("No errors on the demo test set to visualise.")



## Feature Engineering — Correlation & Redundancy

Before finalizing features, check for strongly correlated / redundant
features within each group and drop or flag ones that add noise rather
than signal (kept minimal here since the two feature sets are already
fairly compact and mostly non-redundant by construction).


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.heatmap(font_features_df[FONT_SIZE_FEATURE_NAMES].corr(), annot=True, fmt=".2f",
            cmap="coolwarm", center=0, ax=axes[0], annot_kws={"size": 7})
axes[0].set_title("Font-Size Feature Correlation")

sns.heatmap(readability_features_df[READABILITY_FEATURE_NAMES].corr(), annot=True, fmt=".2f",
            cmap="coolwarm", center=0, ax=axes[1], annot_kws={"size": 7})
axes[1].set_title("Readability Feature Correlation")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "feature_correlation.png", dpi=120)
plt.show()


In [ ]:

def find_highly_correlated_pairs(df, feature_names, threshold=0.92):
    corr = df[feature_names].corr().abs()
    pairs = []
    for i in range(len(feature_names)):
        for j in range(i + 1, len(feature_names)):
            v = corr.iloc[i, j]
            if v >= threshold:
                pairs.append((feature_names[i], feature_names[j], round(float(v), 3)))
    return pairs


print("Highly correlated (>= 0.92) font-size feature pairs:")
print(find_highly_correlated_pairs(font_features_df, FONT_SIZE_FEATURE_NAMES) or "None found.")
print()
print("Highly correlated (>= 0.92) readability feature pairs:")
print(find_highly_correlated_pairs(readability_features_df, READABILITY_FEATURE_NAMES) or "None found.")
print()
print("Note: `contrast_std` and `grayscale_std` are defined identically here")
print("(both = grayscale standard deviation) -- kept as two names for spec")
print("traceability (Section 8 vs Section 12 naming), but only one is")
print("functionally needed; safe to drop one in a production feature set.")



## Image Quality Experiments — Robustness Check

Controlled transformations applied to a few REAL-pipeline crops (from the
demo images, for illustration) to see how readability features and the
trained model respond as quality degrades. This is **augmentation /
robustness testing only** — never a substitute for real human-labelled
data, and these synthetic variants are not added to the training set used
above.


In [ ]:

sample_row = dataset_df.iloc[0]
base_img = load_image_bgr(sample_row["image_path"])
base_crop = crop_text_region(base_img, (sample_row["bbox_x1"], sample_row["bbox_y1"],
                                          sample_row["bbox_x2"], sample_row["bbox_y2"]))
base_crop_pil = Image.fromarray(cv2.cvtColor(base_crop, cv2.COLOR_BGR2RGB))

transform_modes = ["none", "blur", "low_contrast", "noise", "downscale", "jpeg"]
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
robustness_rows = []

for ax, mode in zip(axes.flat, transform_modes):
    degraded = _degrade_image(base_crop_pil, mode)
    gray = cv2.cvtColor(degraded, cv2.COLOR_BGR2GRAY) if degraded.ndim == 3 else degraded
    sharp = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    contrast = float(gray.std())
    ax.imshow(cv2.cvtColor(degraded, cv2.COLOR_BGR2RGB) if degraded.ndim == 3 else degraded, cmap="gray")
    ax.set_title(f"{mode}\nsharpness={sharp:.0f}  contrast={contrast:.0f}", fontsize=9)
    ax.axis("off")
    robustness_rows.append({"transform": mode, "sharpness": sharp, "contrast": contrast})

plt.suptitle("SYNTHETIC AUGMENTATION -- robustness illustration only", y=1.03)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "image_quality_robustness.png", dpi=120, bbox_inches="tight")
plt.show()

pd.DataFrame(robustness_rows)



## Human Annotation Workflow

A CSV-based annotation workflow (robust in Colab, where interactive
widgets can be unreliable across environments). This displays each crop
and appends the annotator's response to a CSV following the Section 5
schema. **No fake labels are generated by this workflow** — it always
produces an empty template plus whatever a human types.


In [ ]:

ANNOTATION_CSV = ANNOTATIONS_DIR / "readability_font_annotations.csv"


def init_annotation_csv(df, path=ANNOTATION_CSV):
    template = df[["image_id", "package_id"]].copy()
    for col in ["readability_label", "readability_score", "actual_font_height_mm",
                "measurement_method", "annotator_id", "annotation_quality", "annotation_notes"]:
        template[col] = None
    if not path.exists():
        template.to_csv(path, index=False)
        print(f"Created empty annotation template at: {path}")
    else:
        print(f"Annotation file already exists at: {path} (not overwritten)")
    return template


def show_crop_for_annotation(row_or_image_id, df=dataset_df):
    if isinstance(row_or_image_id, str):
        row = df[df["image_id"] == row_or_image_id].iloc[0]
    else:
        row = row_or_image_id
    img = load_image_bgr(row["image_path"])
    crop = crop_text_region(img, (row["bbox_x1"], row["bbox_y1"], row["bbox_x2"], row["bbox_y2"]))
    plt.figure(figsize=(5, 3))
    plt.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    plt.title(f"{row['image_id']} | OCR text: \"{row['text']}\"")
    plt.axis("off")
    plt.show()
    return row


def save_annotation(image_id, readability_label=None, readability_score=None,
                     actual_font_height_mm=None, measurement_method=None,
                     annotator_id=None, annotation_quality=None, annotation_notes=None,
                     path=ANNOTATION_CSV):
    ann_df = pd.read_csv(path) if path.exists() else init_annotation_csv(dataset_df, path)
    row_mask = ann_df["image_id"] == image_id
    if not row_mask.any():
        raise ValueError(f"image_id {image_id} not found in annotation template")
    for col, val in [
        ("readability_label", readability_label), ("readability_score", readability_score),
        ("actual_font_height_mm", actual_font_height_mm), ("measurement_method", measurement_method),
        ("annotator_id", annotator_id), ("annotation_quality", annotation_quality),
        ("annotation_notes", annotation_notes),
    ]:
        if val is not None:
            ann_df.loc[row_mask, col] = val
    ann_df.to_csv(path, index=False)
    return ann_df.loc[row_mask]


# Demonstration of the workflow's mechanics (NOT a real annotation -- for a
# real project, a human reviews `show_crop_for_annotation(...)` output and
# calls `save_annotation(...)` with THEIR OWN judgement).
init_annotation_csv(dataset_df)
_ = show_crop_for_annotation(dataset_df.iloc[0]["image_id"])
print("\nTo annotate for real: call show_crop_for_annotation(image_id), look at")
print("the crop yourself, then call save_annotation(image_id, readability_label=...,")
print("actual_font_height_mm=..., annotator_id='your_id', ...).")



# COMBINED

## 21. Combined Font Size + Readability Inference
## 22. Manual Verification Logic

`analyze_text_region()` combines the two **independent** analyses into one
structured output. It:

- Never fabricates a probability where the model only gives an uncalibrated
  score (labels it `model_score` instead of `probability` in that case).
- Routes to `MANUAL_VERIFICATION_REQUIRED` whenever calibration is missing,
  image quality is too poor to trust the readability features, or model
  confidence is low.
- **Never** outputs a legal verdict. The output is measurement + status
  only — NiriKsha's separate rule engine consumes this to make the actual
  compliance decision.


In [ ]:

# Minimum resolution below which we don't trust ANY readability feature
# enough to report a model-backed status -- a measurable image-quality gate,
# not a legal readability standard.
MIN_TRUSTED_CROP_PIXELS = 200  # a crop smaller than ~14x14px carries almost no signal

# Below this max predicted-class probability, route to manual verification
# rather than trust the model's own decision.
MIN_MODEL_CONFIDENCE = 0.55


def analyze_text_region(row, font_model, font_scaler, font_model_name,
                         readability_model, readability_scaler, readability_model_name,
                         image_bgr=None):
    '''Combined Font Size + Readability analysis for ONE text region.
    Returns a JSON-serialisable dict. NEVER returns a legal verdict.'''
    if image_bgr is None:
        image_bgr = load_image_bgr(row["image_path"])

    # ---- Font size ----
    font_feats = extract_font_size_features(row, image_bgr)
    calibration_info = {
        "pixels_per_mm": row.get("pixels_per_mm"),
        "calibration_source": row.get("calibration_source"),
    }
    font_result = estimate_font_height(font_feats["pixel_height"], calibration_info)
    if font_result["status"] == "ESTIMATED":
        Xf = pd.DataFrame([font_feats])[FONT_SIZE_FEATURE_NAMES]
        Xf_in = font_scaler.transform(Xf) if font_model_name == "LinearRegression" else Xf
        font_result["height_mm"] = round(float(font_model.predict(Xf_in)[0]), 3)
        font_result["method"] = f"ML model ({font_model_name})"

    # ---- Readability ----
    read_feats = extract_readability_features(row, image_bgr)
    requires_manual = False
    if read_feats["crop_resolution"] < MIN_TRUSTED_CROP_PIXELS:
        readability_result = {"status": "READABILITY_UNCERTAIN",
                               "reason": "crop resolution too low to trust visual features"}
        requires_manual = True
    else:
        Xr = pd.DataFrame([read_feats])[READABILITY_FEATURE_NAMES]
        Xr_in = readability_scaler.transform(Xr) if readability_model_name == "LogisticRegression" else Xr
        if readability_model_name == "XGBClassifier":
            pred_int = readability_model.predict(Xr_in)[0]
            pred_label = {v: k for k, v in _label_to_int.items()}[pred_int]
        else:
            pred_label = readability_model.predict(Xr_in)[0]

        confidence = None
        confidence_kind = None
        if hasattr(readability_model, "predict_proba"):
            proba = readability_model.predict_proba(Xr_in)[0]
            confidence = float(proba.max())
            confidence_kind = "probability"  # model exposes calibrated-ish predict_proba
        else:
            confidence = None
            confidence_kind = None

        if confidence is not None and confidence < MIN_MODEL_CONFIDENCE:
            readability_result = {
                "status": "MANUAL_VERIFICATION_REQUIRED",
                "predicted_label": pred_label,
                confidence_kind: round(confidence, 3),
                "reason": f"model confidence {confidence:.3f} below threshold {MIN_MODEL_CONFIDENCE}",
            }
            requires_manual = True
        else:
            readability_result = {"status": pred_label}
            if confidence is not None:
                readability_result[confidence_kind] = round(confidence, 3)

    if font_result["status"] == "FONT_SIZE_UNDETERMINABLE":
        requires_manual = True

    return {
        "image_id": row["image_id"],
        "text": row["text"],
        "font_size": font_result,
        "readability": readability_result,
        "requires_manual_verification": requires_manual,
    }


# Demo run over a handful of test rows
demo_infer_df = dataset_df.merge(demo_calibration_df, on="package_id", how="left")
example_outputs = []
for _, r in demo_infer_df.sample(5, random_state=SEED).iterrows():
    out = analyze_text_region(
        r, best_font_model, font_size_scaler, best_font_model_name,
        best_readability_model, readability_scaler, best_readability_model_name,
    )
    example_outputs.append(out)
    print(json.dumps(out, indent=2, default=str))
    print("-" * 60)



## Testing

Edge-case tests for both analyses. The critical property under test:
**uncertain cases must route to `FONT_SIZE_UNDETERMINABLE` /
`READABILITY_UNCERTAIN` / `MANUAL_VERIFICATION_REQUIRED`**, never to a
confident (and possibly wrong) determination.


In [ ]:

def _mk_row(**overrides):
    base = dict(
        image_id="TEST", package_id="PKG_TEST", image_path=dataset_df.iloc[0]["image_path"],
        bbox_x1=10, bbox_y1=10, bbox_x2=60, bbox_y2=40, text="TEST TEXT",
        declaration_type="GENERIC", ocr_confidence=0.9,
        pixels_per_mm=10.0, calibration_source="test",
    )
    base.update(overrides)
    return pd.Series(base)


test_results = []

def check(name, condition):
    test_results.append((name, bool(condition)))
    print(("PASS" if condition else "FAIL"), "-", name)


# ---- Font size tests ----
img0 = load_image_bgr(dataset_df.iloc[0]["image_path"])

row_valid_bbox = _mk_row(bbox_x1=10, bbox_y1=10, bbox_x2=60, bbox_y2=40)
feats = extract_font_size_features(row_valid_bbox, img0)
check("valid bbox -> positive pixel_height", feats["pixel_height"] > 0)

row_invalid_bbox = _mk_row(bbox_x1=60, bbox_y1=40, bbox_x2=10, bbox_y2=10)  # x2<x1, y2<y1
bad_w = row_invalid_bbox["bbox_x2"] - row_invalid_bbox["bbox_x1"]
check("invalid bbox is detectable (negative width)", bad_w < 0)

r_missing_cal = estimate_font_height(30, {"pixels_per_mm": None})
check("missing calibration -> FONT_SIZE_UNDETERMINABLE", r_missing_cal["status"] == "FONT_SIZE_UNDETERMINABLE")

r_invalid_cal = estimate_font_height(30, {"pixels_per_mm": -5})
check("invalid (negative) calibration -> FONT_SIZE_UNDETERMINABLE", r_invalid_cal["status"] == "FONT_SIZE_UNDETERMINABLE")

r_valid_cal = estimate_font_height(30, {"pixels_per_mm": 10.0})
check("valid calibration -> ESTIMATED with a numeric height", r_valid_cal["status"] == "ESTIMATED" and r_valid_cal["height_mm"] is not None)

try:
    extract_font_size_features(_mk_row(text=None), img0)
    check("missing text handled gracefully (no crash)", True)
except Exception:
    check("missing text handled gracefully (no crash)", False)

# ---- Readability tests ----
# Use a real text bbox from the dataset (not arbitrary coordinates) so the
# crop actually contains printed text and these checks are meaningful.
_text_row0 = dataset_df.iloc[0]
_text_img0 = load_image_bgr(_text_row0["image_path"])
sharp_crop = crop_text_region(
    _text_img0, (_text_row0["bbox_x1"], _text_row0["bbox_y1"],
                 _text_row0["bbox_x2"], _text_row0["bbox_y2"])
)
blurry_crop = cv2.GaussianBlur(sharp_crop, (9, 9), 0)
blur_sharp = float(cv2.Laplacian(cv2.cvtColor(blurry_crop, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var())
sharp_sharp = float(cv2.Laplacian(cv2.cvtColor(sharp_crop, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var())
check("blurry crop has lower Laplacian variance than sharp crop", blur_sharp < sharp_sharp)

low_c_crop = _degrade_image(Image.fromarray(cv2.cvtColor(sharp_crop, cv2.COLOR_BGR2RGB)), "low_contrast")
high_c_std = float(cv2.cvtColor(sharp_crop, cv2.COLOR_BGR2GRAY).std())
low_c_std = float(cv2.cvtColor(low_c_crop, cv2.COLOR_BGR2GRAY).std())
check("low-contrast transform reduces grayscale std", low_c_std < high_c_std)

row_missing_conf = _mk_row(ocr_confidence=np.nan)
try:
    f = extract_readability_features(row_missing_conf, img0)
    check("missing OCR confidence handled without crashing", True)
except Exception:
    check("missing OCR confidence handled without crashing", False)

tiny_row = _mk_row(bbox_x1=10, bbox_y1=10, bbox_x2=12, bbox_y2=11)  # ~2x1 px region
tiny_feats = extract_readability_features(tiny_row, img0)
out_tiny = {"crop_resolution": tiny_feats["crop_resolution"]}
check("insufficient resolution is flagged (< MIN_TRUSTED_CROP_PIXELS)",
      tiny_feats["crop_resolution"] < MIN_TRUSTED_CROP_PIXELS)

# ---- End-to-end: uncertain cases must route to manual verification ----
uncertain_row = _mk_row(pixels_per_mm=None, bbox_x1=10, bbox_y1=10, bbox_x2=12, bbox_y2=11)
uncertain_out = analyze_text_region(
    uncertain_row, best_font_model, font_size_scaler, best_font_model_name,
    best_readability_model, readability_scaler, best_readability_model_name,
    image_bgr=img0,
)
check("uncertain case (no calibration + tiny crop) -> requires_manual_verification=True",
      uncertain_out["requires_manual_verification"] is True)
check("uncertain case is NOT silently classified as a violation (no such field exists)",
      "legal_violation" not in uncertain_out and "compliance" not in uncertain_out)

n_pass = sum(1 for _, ok in test_results if ok)
print(f"\n{n_pass}/{len(test_results)} tests passed.")
assert n_pass == len(test_results), "Some tests failed -- see output above."



## 23. Model Export

Exports trained models, feature schemas, and metadata for later integration
into NiriKsha. Metrics in the metadata come **directly from the evaluation
cells above** — nothing here is invented, and everything is explicitly
tagged as demo-data-derived.


In [ ]:

font_model_path = MODELS_DIR / "font_size_model.joblib"
readability_model_path = MODELS_DIR / "readability_model.joblib"
font_scaler_path = MODELS_DIR / "font_size_scaler.joblib"
readability_scaler_path = MODELS_DIR / "readability_scaler.joblib"

joblib.dump(best_font_model, font_model_path)
joblib.dump(best_readability_model, readability_model_path)
joblib.dump(font_size_scaler, font_scaler_path)
joblib.dump(readability_scaler, readability_scaler_path)

with open(MODELS_DIR / "font_size_features.json", "w") as f:
    json.dump({"features": FONT_SIZE_FEATURE_NAMES, "target": "actual_font_height_mm"}, f, indent=2)

with open(MODELS_DIR / "readability_features.json", "w") as f:
    json.dump({"features": READABILITY_FEATURE_NAMES, "target": "readability_label",
               "label_order": LABEL_ORDER}, f, indent=2)

print("Saved:")
for p in [font_model_path, readability_model_path, font_scaler_path, readability_scaler_path]:
    print(" -", p)


In [ ]:

model_metadata = {
    "notebook_scope": "Font Size & Readability Analysis ONLY (NiriKsha, SIH 2026)",
    "data_status": "DEMONSTRATION DATA -- NOT REAL-WORLD TRAINING DATA. "
                    "Replace with real, human-annotated package images before "
                    "trusting any metric below.",
    "generated_at_utc": dt.datetime.utcnow().isoformat() + "Z",
    "dataset_version": "demo-v1-synthetic",
    "n_packages_total": int(dataset_df["package_id"].nunique()),
    "n_packages_train": int(train_df["package_id"].nunique()),
    "n_packages_val": int(val_df["package_id"].nunique()),
    "n_packages_test": int(test_df["package_id"].nunique()),
    "font_size_model": {
        "model_type": best_font_model_name,
        "features": FONT_SIZE_FEATURE_NAMES,
        "requires_scaling": best_font_model_name == "LinearRegression",
        "test_metrics": font_size_final_metrics,
        "baseline_test_metrics": baseline_final_metrics,
        "cross_validation": {k: {"mean": float(np.mean(v)), "std": float(np.std(v))} for k, v in cv_scores.items()},
        "calibration_requirement": (
            "REQUIRES a valid pixels_per_mm calibration per package/image. "
            "Without it, this model must NOT be invoked -- use "
            "estimate_font_height() which returns FONT_SIZE_UNDETERMINABLE."
        ),
    },
    "readability_model": {
        "model_type": best_readability_model_name,
        "features": READABILITY_FEATURE_NAMES,
        "label_order": LABEL_ORDER,
        "requires_scaling": best_readability_model_name == "LogisticRegression",
        "val_metrics": readability_results_df[readability_results_df["model"] == best_readability_model_name].to_dict("records")[0],
        "test_accuracy": float(ml_test_acc),
        "cv_baseline_test_accuracy": float(cv_baseline_test_acc),
        "cross_validation": {k: {"mean": float(np.mean(v)), "std": float(np.std(v))} for k, v in cv_scores_r.items()},
        "min_trusted_crop_pixels": MIN_TRUSTED_CROP_PIXELS,
        "min_model_confidence_for_auto_decision": MIN_MODEL_CONFIDENCE,
    },
    "known_limitations": [
        "Trained/evaluated on synthetic demo data only -- no real-world accuracy claim.",
        "Readability labels in the demo are a rule-based synthetic proxy, NOT human annotations.",
        "Font-size ground truth in the demo comes from generation-time known values, not physical measurement.",
        "No legal thresholds are encoded anywhere in this notebook or its exported artifacts.",
    ],
}

with open(MODELS_DIR / "model_metadata.json", "w") as f:
    json.dump(model_metadata, f, indent=2, default=str)

print(json.dumps(model_metadata, indent=2, default=str))



## 24. Final Demo

End-to-end run of `analyze_text_region()` over a fresh sample of demo rows,
loading the just-exported models from disk (proving the export/import round
trip works), with the crops shown alongside their combined output.


In [ ]:

loaded_font_model = joblib.load(font_model_path)
loaded_readability_model = joblib.load(readability_model_path)
loaded_font_scaler = joblib.load(font_scaler_path)
loaded_readability_scaler = joblib.load(readability_scaler_path)

final_demo_rows = demo_infer_df.sample(6, random_state=7)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for ax, (_, r) in zip(axes.flat, final_demo_rows.iterrows()):
    img = load_image_bgr(r["image_path"])
    crop = crop_text_region(img, (r["bbox_x1"], r["bbox_y1"], r["bbox_x2"], r["bbox_y2"]))
    result = analyze_text_region(
        r, loaded_font_model, loaded_font_scaler, best_font_model_name,
        loaded_readability_model, loaded_readability_scaler, best_readability_model_name,
        image_bgr=img,
    )

    ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    fs = result["font_size"]
    rd = result["readability"]
    fs_line = f"Font: {fs['status']}" + (f" ({fs['height_mm']}mm)" if fs.get("height_mm") is not None else "")
    rd_line = f"Read: {rd['status']}"
    manual_flag = " [MANUAL REVIEW]" if result["requires_manual_verification"] else ""
    ax.set_title(f"{fs_line}\n{rd_line}{manual_flag}", fontsize=9)
    ax.axis("off")

plt.suptitle("Final Demo -- Combined Font Size + Readability Analysis (DEMO DATA)", y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "final_demo.png", dpi=120, bbox_inches="tight")
plt.show()

print("Reminder: none of these outputs are legal compliance verdicts.")
print("They are handed to NiriKsha's separate, verified rule engine for that decision.")



## 25. Limitations and Next Steps

### Honest limitations of what's in this notebook right now

1. **All metrics above come from synthetic demo data.** They prove the
   pipeline runs end-to-end and is structurally sound; they say nothing
   about real-world accuracy. Do not report them as NiriKsha's real
   performance.
2. **Readability labels are a synthetic, rule-based proxy** for the demo
   only — real deployment requires human-annotated `READABLE` /
   `PARTIALLY_READABLE` / `NOT_READABLE` labels (Human Annotation section).
3. **Font-size ground truth in the demo is generation-time metadata**, not
   a physical caliper/reference measurement — real deployment requires
   actual physical measurements tied to valid calibration.
4. The perspective-distortion indicator and noise estimate are simple,
   explainable proxies, not a full geometric rectification pipeline — fine
   for an explainable SIH-scale system, but worth revisiting if real data
   shows they're insufficient.
5. Class imbalance in `readability_label` was not specifically addressed
   (no resampling/class-weighting) — worth adding once real label
   distributions are known.

### Concrete next steps

1. Replace `demo_df` with a real CSV built from real package photographs,
   following the Section 3 schema, loaded via `load_dataset(csv_path=...)`.
2. Run real human annotation (`show_crop_for_annotation` /
   `save_annotation`) across a representative sample of real declarations.
3. Establish real physical calibration per image/package (reference
   object, known package dimension, or a fixed capture rig) and populate
   `pixels_per_mm` / `calibration_source` for real rows.
4. Re-run Sections 7–20 unchanged — the pipeline itself does not need to
   change, only the data feeding it.
5. Re-evaluate `MIN_TRUSTED_CROP_PIXELS` and `MIN_MODEL_CONFIDENCE`
   against real validation data (these are currently reasonable starting
   points, not tuned values).
6. Only if a real, sufficiently large annotated dataset later shows
   classical ML + CV features are insufficient, consider a small CNN for
   readability (Section 25 of the original spec: don't over-engineer
   before there's a demonstrated reason to).
7. Hand `font_size_model.joblib`, `readability_model.joblib`, and their
   companion JSON schemas/metadata to the NiriKsha integration layer — this
   notebook's job ends at producing validated measurements/status, not at
   deciding compliance.
